In [61]:
import pandas as pd 

import geopandas as gpd
def clean_cols(df):
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
    return df


In [62]:
path = "/Users/amodi/Downloads/"

restaurants = clean_cols(pd.read_csv(path + "DOHMH_New_York_City_Restaurant_Inspection_Results_20260926.csv", low_memory=False))
events      = clean_cols(pd.read_csv(path + "NYC_Permitted_Event_Information_20260926.csv"))
markets     = clean_cols(pd.read_csv(path + "NYC_Farmers_Markets_20260926.csv"))
gardens     = clean_cols(pd.read_csv(path + "GreenThumb_Garden_Info_20260926.csv"))
art         = clean_cols(pd.read_csv(path + "Public_Design_Commission_Outdoor_Public_Art_Inventory_20260926.csv"))
pops        = clean_cols(pd.read_csv(path + "Privately_Owned_Public_Spaces_(POPS)_20260926.csv"))

In [63]:
for name, df in {"Restaurants": restaurants, "events": events, "markets": markets,
                 "gardens": gardens, "art": art, "pops": pops}.items():
    print(name, df.shape)
    print(list(df.columns), "\n")

Restaurants (295186, 27)
['camis', 'dba', 'boro', 'building', 'street', 'zipcode', 'phone', 'cuisine_description', 'inspection_date', 'action', 'violation_code', 'violation_description', 'critical_flag', 'score', 'grade', 'grade_date', 'record_date', 'inspection_type', 'latitude', 'longitude', 'community_board', 'council_district', 'census_tract', 'bin', 'bbl', 'nta', 'location'] 

events (31639, 13)
['event_id', 'event_name', 'start_date_time', 'end_date_time', 'event_agency', 'event_type', 'event_borough', 'event_location', 'event_street_side', 'street_closure_type', 'community_board', 'police_precinct', 'cemsid'] 

markets (2218, 13)
['year', 'market_name', 'borough', 'street_address', 'community_district', 'latitude', 'longitude', 'days_of_operation', 'hours_of_operations', 'season_dates', 'accepts_ebt', 'open_year_round', 'stellar_cooking_demonstrations'] 

gardens (635, 27)
['assemblydist', 'address', 'borough', 'communityboard', 'congressionaldist', 'coundist', 'gardenname', 'ju

In [64]:
ntas = gpd.read_file("https://data.cityofnewyork.us/resource/9nt8-h7nd.geojson?$limit=1000")
ntas = ntas[ntas["ntatype"].astype(str) == "0"]   # residential only
ntas["area_km2"] = ntas.to_crs(2263).area * 0.09290304 / 1e6
print(len(ntas), "neighborhoods")

197 neighborhoods


In [65]:
# one row per restaurant, with a cuisine
restaurants_1 = restaurants.drop_duplicates("camis")
restaurants_1 = restaurants_1[restaurants_1["cuisine_description"].notna()]
print("restaurants:", len(restaurants_1))

# farmers markets: latest year only
markets_1 = markets[markets["year"] == markets["year"].max()]
print("markets:", len(markets_1), "in", markets["year"].max())

# gardens: check status values and coordinate columns
print(gardens["status"].value_counts())
print([c for c in gardens.columns if "lat" in c or "lon" in c])
print([c for c in art.columns if "lat" in c or "lon" in c])

restaurants: 27545
markets: 103 in 2026
status
Active                      547
Not GreenThumb               67
Inactive (No Group)           8
Closed (Construction)         6
Active (Unlicensed)           4
Inactive (Group Forming)      3
Name: count, dtype: int64
['lat', 'lon']
['latitude', 'longitude']


In [66]:
gardens_1 = gardens[gardens["status"].isin(["Active", "Active (Unlicensed)"])]
print("gardens:", len(gardens_1))

def count_per_nta(df, name, lat="latitude", lon="longitude"):
    df = df.copy()
    df[lat] = pd.to_numeric(df[lat], errors="coerce")
    df[lon] = pd.to_numeric(df[lon], errors="coerce")
    df = df[df[lat].between(40.4, 41.0) & df[lon].between(-74.3, -73.6)]
    pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs="EPSG:4326")
    joined = gpd.sjoin(pts, ntas[["nta2020", "geometry"]], predicate="within")
    return joined.groupby("nta2020").size().rename(name)

gardens: 551


In [67]:
counts = pd.concat([
    count_per_nta(restaurants_1, "restaurants"),
    count_per_nta(markets_1, "farmers_markets"),
    count_per_nta(gardens_1, "gardens", lat="lat", lon="lon"),
    count_per_nta(art, "public_art"),
    # count_per_nta(pops, "pops"),   # add back after loading the real POPS file
], axis=1).fillna(0)

counts = counts.join(ntas.set_index("nta2020")[["ntaname", "area_km2"]])
counts.sort_values("restaurants", ascending=False).head(10)

,restaurants,farmers_markets,gardens,public_art,ntaname,area_km2
nta2020,,,,,,
MN0502,1499,1.0,0.0,17.0,Midtown-Times Square,2.281006
MN0501,824,1.0,0.0,21.0,Midtown South-Flatiron-Union Square,1.382306
MN0303,669,1.0,41.0,8.0,East Village,1.763311
MN0401,608,1.0,0.0,3.0,Chelsea-Hudson Yards,2.756596
MN0402,542,1.0,2.0,5.0,Hell's Kitchen,1.707773
QN0707,508,1.0,0.0,6.0,Flushing-Willets Point,3.579952
MN0101,507,2.0,1.0,15.0,Financial District-Battery Park City,1.786223
MN0604,495,1.0,0.0,2.0,East Midtown-Turtle Bay,1.220614
BK0102,485,2.0,9.0,6.0,Williamsburg,2.680513


In [68]:
r = restaurants_1.copy()
r["latitude"] = pd.to_numeric(r["latitude"], errors="coerce")
r["longitude"] = pd.to_numeric(r["longitude"], errors="coerce")
r = r[r["latitude"].between(40.4, 41.0) & r["longitude"].between(-74.3, -73.6)]
r = r[r["cuisine_description"] != "Not Listed/Not Applicable"]
r_pts = gpd.GeoDataFrame(r, geometry=gpd.points_from_xy(r["longitude"], r["latitude"]), crs="EPSG:4326")
r_pts = gpd.sjoin(r_pts, ntas[["nta2020", "geometry"]], predicate="within")

g = r_pts.groupby("nta2020")["cuisine_description"]
cuisine = pd.DataFrame({
    "cuisine_diversity": g.nunique(),
    "top_cuisines": g.agg(lambda s: ", ".join(s.value_counts().head(3).index)),
})
counts = counts.drop(columns=["cuisine_diversity", "top_cuisines"], errors="ignore").join(cuisine)
counts[["ntaname", "cuisine_diversity", "top_cuisines"]].sort_values("cuisine_diversity", ascending=False).head(10)

,ntaname,cuisine_diversity,top_cuisines
nta2020,,,
MN0501,Midtown South-Flatiron-Union Square,62,"American, Korean, Coffee/Tea"
MN0502,Midtown-Times Square,62,"American, Coffee/Tea, Italian"
MN0303,East Village,62,"American, Coffee/Tea, Japanese"
BK0102,Williamsburg,58,"American, Coffee/Tea, Japanese"
BK0202,Downtown Brooklyn-DUMBO-Boerum Hill,56,"American, Coffee/Tea, Pizza"
MN0101,Financial District-Battery Park City,56,"American, Coffee/Tea, Italian"
MN0402,Hell's Kitchen,54,"American, Coffee/Tea, Japanese"
MN0401,Chelsea-Hudson Yards,54,"American, Coffee/Tea, Italian"
MN0604,East Midtown-Turtle Bay,53,"American, Coffee/Tea, Japanese"


In [69]:
ct = pd.crosstab(r_pts["nta2020"], r_pts["cuisine_description"])   # restaurants per cuisine per neighborhood
share = ct.div(ct.sum(axis=1), axis=0)                             # neighborhood's cuisine share
city = ct.sum() / ct.values.sum()                                  # citywide cuisine share
ratio = share / city                                               # how over-represented each cuisine is

def signature(nta, min_count=5):
    row = ratio.loc[nta][ct.loc[nta] >= min_count]                 # ignore cuisines with only 1-4 places
    return ", ".join(row.sort_values(ascending=False).head(3).index)

counts["signature_cuisines"] = [signature(n) if n in ct.index else "" for n in counts.index]

counts[counts["ntaname"].str.contains("Flushing|Jackson Heights|Sunset Park|Brighton|Harlem", na=False)][
    ["ntaname", "top_cuisines", "signature_cuisines"]]

,ntaname,top_cuisines,signature_cuisines
nta2020,,,
BK0702,Sunset Park (West),"American, Latin American, Mexican","Latin American, Mexican, Spanish"
BK0703,Sunset Park (Central),"Chinese, Bakery Products/Desserts, Coffee/Tea","Chinese, Asian/Asian Fusion, Bakery Products/D..."
BK1201,Sunset Park (East)-Borough Park (West),"Chinese, Bakery Products/Desserts, Coffee/Tea","Chinese, Bakery Products/Desserts, Coffee/Tea"
BK1303,Brighton Beach,"Eastern European, Turkish, Russian","Russian, Eastern European, Turkish"
MN0902,Manhattanville-West Harlem,"American, Mexican, Latin American",American
MN1001,Harlem (South),"American, Coffee/Tea, Chicken","African, Hamburgers, Chicken"
MN1002,Harlem (North),"Chinese, American, Coffee/Tea","Soul Food, Chicken, Caribbean"
MN1101,East Harlem (South),"American, Mexican, Chinese","Thai, Mexican, Hamburgers"
MN1102,East Harlem (North),"Mexican, Chinese, American","Mexican, Chicken, Other"


In [70]:
generic = ["American", "Coffee/Tea", "Other", "Bottled Beverages", "Hamburgers", "Chicken",
           "Pizza", "Sandwiches", "Donuts", "Juice, Smoothies, Fruit Salads", "Hotdogs"]

def signature(nta, min_count=5):
    row = ratio.loc[nta][(ct.loc[nta] >= min_count) & (~ratio.columns.isin(generic))]
    return ", ".join(row.sort_values(ascending=False).head(3).index)

counts["signature_cuisines"] = [signature(n) if n in ct.index else "" for n in counts.index]

In [71]:
import requests, time

community = ["Block Party", "Street Event", "Street Festival", "Single Block Festival", "Parade",
             "Plaza Event", "Plaza Partner Event", "Open Street Partner Event", "Sidewalk Sale", "Religious Event"]
ev = events[events["event_type"].isin(community)].copy()
print("community events:", len(ev))

def geosearch(text):
    try:
        f = requests.get("https://geosearch.planninglabs.nyc/v2/search",
                         params={"text": text, "size": 1}, timeout=20).json()["features"]
        if f:
            lon, lat = f[0]["geometry"]["coordinates"]
            return lat, lon
    except Exception:
        pass
    return None, None

locs = ev[["event_location", "event_borough"]].drop_duplicates("event_location")
print("unique locations to geocode:", len(locs))
coords = {}
for loc, boro in locs.itertuples(index=False):
    coords[loc] = geosearch(f"{str(loc).split(':')[0]}, {boro}")
    time.sleep(0.1)

ev["latitude"] = ev["event_location"].map(lambda x: coords[x][0])
ev["longitude"] = ev["event_location"].map(lambda x: coords[x][1])
print("geocoded:", round(ev["latitude"].notna().mean() * 100), "%")

community events: 616
unique locations to geocode: 355
geocoded: 24 %


In [72]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

count_cols = ["restaurants", "farmers_markets", "gardens", "public_art"]   # add "pops" once fixed
for c in count_cols:
    counts[f"{c}_per_km2"] = counts[c] / counts["area_km2"]

features = [f"{c}_per_km2" for c in count_cols] + ["cuisine_diversity"]
X = StandardScaler().fit_transform(np.log1p(counts[features].fillna(0)))

for k in range(4, 9):
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X)
    print(k, round(silhouette_score(X, labels), 3))

4 0.27
5 0.278
6 0.294
7 0.247
8 0.254


In [73]:
import os
[f for f in os.listdir(path) if "POPS" in f or "Privately" in f]

['Privately_Owned_Public_Spaces_(POPS)_20260926.csv']

In [74]:
pops = clean_cols(pd.read_csv(path + "Privately_Owned_Public_Spaces_(POPS)_20260926.csv"))
print(pops.shape)
print([c for c in pops.columns if "lat" in c or "lon" in c])

counts = counts.drop(columns=["pops"], errors="ignore").join(count_per_nta(pops, "pops")).fillna({"pops": 0})
counts.sort_values("pops", ascending=False)[["ntaname", "pops"]].head(5)

(392, 34)
['latitude', 'longitude']


,ntaname,pops
nta2020,,
MN0502,Midtown-Times Square,78.0
MN0604,East Midtown-Turtle Bay,50.0
MN0101,Financial District-Battery Park City,49.0
MN0603,Murray Hill-Kips Bay,39.0
MN0803,Upper East Side-Yorkville,30.0


In [75]:
count_cols = ["restaurants", "farmers_markets", "gardens", "public_art", "pops"]

In [76]:
k = 6
counts["cluster"] = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X)

summary = counts.groupby("cluster")[features].mean().round(1)
summary["n_neighborhoods"] = counts["cluster"].value_counts().sort_index()
display(summary)

for c, grp in counts.groupby("cluster"):
    print(f"cluster {c}:", ", ".join(grp["ntaname"].head(8)))

,restaurants_per_km2,farmers_markets_per_km2,gardens_per_km2,public_art_per_km2,cuisine_diversity,n_neighborhoods
cluster,,,,,,
0,35.0,0.1,5.4,0.6,18.1,18
1,58.8,0.9,0.6,1.1,20.7,17
2,17.3,0.1,0.2,0.3,19.0,91
3,75.9,0.1,0.6,0.9,35.6,42
4,300.9,0.6,0.5,6.9,49.4,17
5,104.7,1.1,7.0,3.4,34.4,12


cluster 0: Bedford-Stuyvesant (West), Bedford-Stuyvesant (East), East New York (North), East New York-New Lots, East New York-City Line, Ocean Hill, Brownsville, Mott Haven-Port Morris
cluster 1: Sunset Park (Central), Crown Heights (South), East Flatbush-Erasmus, East Flatbush-Remsen Village, Mount Eden-Claremont (West), University Heights (North)-Fordham, Bedford Park, Norwood
cluster 2: South Williamsburg, Cypress Hills, Spring Creek-Starrett City, Dyker Heights, Bath Beach, Sunset Park (East)-Borough Park (West), Borough Park, Kensington
cluster 3: Greenpoint, East Williamsburg, Clinton Hill, Bushwick (West), Carroll Gardens-Cobble Hill-Gowanus-Red Hook, Windsor Terrace-South Slope, Sunset Park (West), Prospect Heights
cluster 4: Brooklyn Heights, Downtown Brooklyn-DUMBO-Boerum Hill, Park Slope, Financial District-Battery Park City, Tribeca-Civic Center, SoHo-Little Italy-Hudson Square, Greenwich Village, West Village
cluster 5: Williamsburg, Fort Greene, Bushwick (East), Melrose, 

In [77]:
count_cols = ["restaurants", "farmers_markets", "gardens", "public_art", "pops"]
for c in count_cols:
    counts[f"{c}_per_km2"] = counts[c] / counts["area_km2"]

features = [f"{c}_per_km2" for c in count_cols] + ["cuisine_diversity"]
X = StandardScaler().fit_transform(np.log1p(counts[features].fillna(0)))

for k in range(4, 9):
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X)
    print(k, round(silhouette_score(X, labels), 3))

4 0.251
5 0.268
6 0.273
7 0.227
8 0.224


In [78]:
cluster_names = {
    0: "residential with community gardens",
    1: "neighborhood markets and local food",
    2: "quiet residential",
    3: "mid-density with varied food",
    4: "dense downtown with plazas and public art",
    5: "green, community-run, lively food",
}
counts["cluster_name"] = counts["cluster"].map(cluster_names)
counts[["ntaname", "cluster_name"]].sample(10)

,ntaname,cluster_name
nta2020,,
QN1003,Howard Beach-Lindenwood,quiet residential
QN1401,Far Rockaway-Bayswater,quiet residential
BX0501,University Heights (South)-Morris Heights,quiet residential
BX0403,Mount Eden-Claremont (West),neighborhood markets and local food
MN0902,Manhattanville-West Harlem,neighborhood markets and local food
QN0904,Ozone Park (North),quiet residential
MN0604,East Midtown-Turtle Bay,dense downtown with plazas and public art
BK1801,Flatlands,quiet residential
QN0401,Elmhurst,mid-density with varied food


In [79]:
pct = counts[features].rank(pct=True)
labels = {
    "restaurants_per_km2": "restaurant density",
    "cuisine_diversity": "variety of cuisines",
    "farmers_markets_per_km2": "farmers markets",
    "gardens_per_km2": "community gardens",
    "public_art_per_km2": "outdoor public art",
    "pops_per_km2": "public plazas",
}

def level(p):
    return ("very high" if p >= 0.8 else "high" if p >= 0.6 else
            "average" if p >= 0.4 else "low" if p >= 0.2 else "very low")

def describe(row):
    parts = [f"{labels[f]}: {level(pct.loc[row.name, f])}" for f in features]
    text = f"{row['ntaname']}. Type: {row['cluster_name']}. " + "; ".join(parts) + "."
    if row["signature_cuisines"]:
        text += f" Signature food: {row['signature_cuisines']}."
    return text

counts["profile_text"] = counts.apply(describe, axis=1)
print(counts.loc[counts["ntaname"] == "Jackson Heights", "profile_text"].iloc[0])

counts.to_csv("neighborhood_profiles.csv")
print("saved", len(counts), "neighborhoods")

Jackson Heights. Type: mid-density with varied food. restaurant density: very high; farmers markets: high; community gardens: low; outdoor public art: very low; public plazas: average; variety of cuisines: very high. Signature food: Bangladeshi, Peruvian, Latin American.
saved 197 neighborhoods


In [80]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [81]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
profile_vecs = embedder.encode(counts["profile_text"].tolist(), normalize_embeddings=True)
print(profile_vecs.shape)   # (197, 384)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(197, 384)


In [82]:
def match(user_text, k=3):
    q = embedder.encode(user_text, normalize_embeddings=True)
    scores = util.cos_sim(q, profile_vecs)[0].numpy()
    top = scores.argsort()[::-1][:k]
    return counts.iloc[top][["ntaname", "cluster_name", "signature_cuisines"]].assign(score=scores[top].round(3))

display(match("I miss lively evening streets, small food shops, and places where people linger."))
display(match("I grew up by the sea, a slow place with bakeries and old people on benches."))
display(match("Mi barrio en Lima: mercados, comida peruana, mucha gente en la calle."))   # Spanish test

,ntaname,cluster_name,signature_cuisines,score
nta2020,,,,
MN0502,Midtown-Times Square,dense downtown with plazas and public art,"Salads, Steakhouse, Soups/Salads/Sandwiches",0.449
QN0102,Old Astoria-Hallets Point,quiet residential,,0.444
QN0803,Fresh Meadows-Utopia,quiet residential,Jewish/Kosher,0.435


,ntaname,cluster_name,signature_cuisines,score
nta2020,,,,
BK1802,Marine Park-Mill Basin-Bergen Beach,quiet residential,"Bagels/Pretzels, Frozen Desserts, Chinese",0.398
SI0202,New Dorp-Midland Beach,quiet residential,"Frozen Desserts, Chinese, Japanese",0.388
SI0107,Mariner's Harbor-Arlington-Graniteville,quiet residential,,0.354


,ntaname,cluster_name,signature_cuisines,score
nta2020,,,,
BX0401,Concourse-Concourse Village,"green, community-run, lively food","Spanish, Caribbean, Latin American",0.613
QN1201,Jamaica,mid-density with varied food,"Bangladeshi, Spanish, Caribbean",0.581
BK1704,East Flatbush-Remsen Village,neighborhood markets and local food,"Caribbean, Chinese",0.574


In [83]:
phrases = {
    "restaurants_per_km2": "streets packed with restaurants and small food shops",
    "cuisine_diversity": "food from many different cultures",
    "farmers_markets_per_km2": "open-air farmers markets",
    "gardens_per_km2": "community gardens run by neighbors",
    "public_art_per_km2": "outdoor sculptures, monuments and public art",
    "pops_per_km2": "busy plazas and public seating among tall office buildings",
}

def describe_natural(row):
    has = [phrases[f] for f in features if pct.loc[row.name, f] >= 0.7]
    if pct.loc[row.name, "restaurants_per_km2"] < 0.3:
        has.append("quiet, mostly residential streets")
    text = "A neighborhood with " + ", ".join(has) + "." if has else "A quiet residential neighborhood."
    if row["signature_cuisines"]:
        text += f" Known for {row['signature_cuisines']} food."
    return text

counts["embed_text"] = counts.apply(describe_natural, axis=1)
print(counts.loc[counts["ntaname"] == "Jackson Heights", "embed_text"].iloc[0])

A neighborhood with streets packed with restaurants and small food shops, food from many different cultures. Known for Bangladeshi, Peruvian, Latin American food.


In [84]:
extra = {
    "Brighton Beach": "By the ocean with a boardwalk and beach. Large Russian and Ukrainian community.",
    "Jackson Heights": "Very busy evening sidewalks with street vendors. Large South Asian and Latin American communities.",
    # add your other demo neighborhoods here
}
for name, tag in extra.items():
    counts.loc[counts["ntaname"] == name, "embed_text"] += " " + tag

In [85]:
profile_vecs = embedder.encode(counts["embed_text"].tolist(), normalize_embeddings=True)

In [86]:
counts.to_csv("neighborhood_profiles.csv")
np.save("profile_vecs.npy", profile_vecs)

In [87]:
main_cols = [
    "ntaname", "area_km2", "cluster", "cluster_name",
    "restaurants", "farmers_markets", "gardens", "public_art", "pops",
    "restaurants_per_km2", "farmers_markets_per_km2", "gardens_per_km2",
    "public_art_per_km2", "pops_per_km2",
    "cuisine_diversity", "signature_cuisines",
    "profile_text", "embed_text",
]
summary = counts[main_cols].round(2)
summary.to_csv("neighborhood_summary.csv")
summary.head()

,ntaname,area_km2,cluster,cluster_name,restaurants,farmers_markets,gardens,public_art,pops,restaurants_per_km2,farmers_markets_per_km2,gardens_per_km2,public_art_per_km2,pops_per_km2,cuisine_diversity,signature_cuisines,profile_text,embed_text
nta2020,,,,,,,,,,,,,,,,,,
BK0101,Greenpoint,3.28,3,mid-density with varied food,262,1.0,3.0,4.0,0.0,79.84,0.30,0.91,1.22,0.00,45,"Polish, Thai, Japanese",Greenpoint. Type: mid-density with varied food...,A neighborhood with streets packed with restau...
BK0102,Williamsburg,2.68,5,"green, community-run, lively food",485,2.0,9.0,6.0,2.0,180.94,0.75,3.36,2.24,0.75,58,"New American, French, Middle Eastern","Williamsburg. Type: green, community-run, live...",A neighborhood with streets packed with restau...
BK0103,South Williamsburg,1.41,2,quiet residential,22,0.0,1.0,1.0,1.0,15.57,0.00,0.71,0.71,0.71,11,Jewish/Kosher,South Williamsburg. Type: quiet residential. r...,A neighborhood with busy plazas and public sea...
BK0104,East Williamsburg,4.86,3,mid-density with varied food,288,0.0,6.0,4.0,0.0,59.31,0.00,1.24,0.82,0.00,52,"Mediterranean, Thai, Mexican",East Williamsburg. Type: mid-density with vari...,A neighborhood with streets packed with restau...
BK0201,Brooklyn Heights,0.93,4,dense downtown with plazas and public art,129,1.0,1.0,4.0,1.0,139.10,1.08,1.08,4.31,1.08,33,"New American, Japanese, Italian",Brooklyn Heights. Type: dense downtown with pl...,A neighborhood with streets packed with restau...


In [92]:
count.isnull().sum()

NameError: name 'count' is not defined